# Baseline

In this notebook, we are going to learn how to use Meta's large pre-trained model [NLLB](https://huggingface.co/docs/transformers/model_doc/nllb) on the [MultiUN dataset](https://huggingface.co/datasets/Helsinki-NLP/multiun), specifically the Russian to Chinese (ru-zh) subset.

In [ ]:
%pip install datasets evaluate transformers accelerate peft bitsandbytes huggingface_hub
%pip install sacrebleu
%pip install unbabel-comet

## Authentication
Please authenticate with Hugging Face to access the dataset and model.

In [ ]:
import os
from huggingface_hub import login

# Authenticate using token from environment variable
# To set the token, use: export HF_TOKEN="your_token_here"
# Or the script will skip authentication if already logged in
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("✓ Authenticated with HuggingFace")
else:
    print("⚠ HF_TOKEN not found, proceeding without authentication")

MultiUN is available in the [Datasets repository](https://huggingface.co/datasets) from Hugging Face. The [Datasets library](https://huggingface.co/docs/datasets) makes easy to access and load datasets. For example, you can easily load your own dataset following [this tutorial](https://huggingface.co/docs/datasets/loading#local-and-remote-files).

In [20]:
from datasets import load_dataset, DatasetDict

# Load the MultiUN dataset for Russian-Chinese
raw_datasets = load_dataset("Helsinki-NLP/multiun", "ru-zh")

# MultiUN only has a train split, so we create validation and test splits
# We'll take a smaller subset for demonstration purposes if the dataset is too large.
# The user requested: 10,000 for training, 1,000 for test, 1,000 for validation.
# Total needed: 12,000.

# Shuffling and selecting a subset of 12,000 total samples
raw_datasets["train"] = raw_datasets["train"].shuffle(seed=42).select(range(12000))

# Splitting: 10,000 for training, 2,000 remaining for validation and test
train_test = raw_datasets["train"].train_test_split(test_size=2000, seed=42)

# Splitting the 2,000 remaining: 1,000 for validation, 1,000 for test
test_val = train_test["test"].train_test_split(test_size=1000, seed=42)

raw_datasets = DatasetDict({
    "train": train_test["train"],
    "validation": test_val["train"],
    "test": test_val["test"]
})

# The dataset has a 'translation' column with 'ru' and 'zh' keys. 
# We map it to 'source_text' and 'dest_text' to match the notebook structure.
def map_to_src_tgt(batch):
    return {
        "source_text": [x["ru"] for x in batch["translation"]],
        "dest_text": [x["zh"] for x in batch["translation"]],
        "dest_lang": ["zh"] * len(batch["translation"]) 
    }

raw_datasets = raw_datasets.map(map_to_src_tgt, batched=True, remove_columns=["translation"])

print(raw_datasets)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 1000
    })
})


We have manually created the training, validation, and test splits from the original training set. Each set is a dictionary with a list of source sentences (source_text), target sentences (dest_text) and the target language (dest_lang).

Let's take a closer look at the features of the training set:

In [21]:
raw_datasets["train"].features

{'source_text': Value('string'),
 'dest_text': Value('string'),
 'dest_lang': Value('string')}

We are focusing on the translation from Russian to Chinese.

Let us take a look at the translations of the first two Russian sentences:

In [22]:
raw_datasets["train"][:14]["source_text"]

['Упрощения текста и подготовки прямых переводов Декларации на различные языки коренных народов будет явно недостаточно, и потребуется принять другие меры для создания потенциала в рамках общин коренных и некоренных народов.',
 'Достаточно назвать в этой связи Международный трибунал по бывшей Югославии, Международный уголовный трибунал по Руанде, Международный трибунал по морскому праву, Международный уголовный суд.',
 'Председатель: За проект резолюции подано 15\xa0голосов.',
 'Вот почему мы должны засучить рукава и решительно взяться за переделку этого органа, заседающего за столом в форме подковы.',
 'i) систематического совпадения проверок и пиков в накоплении углерода; и',
 'В библиотеках содержится почти 60\xa0млн.',
 '- придания эффективного характера трудовому законодательству и трудовым институтам, в том числе в отношении признания трудового правоотношения, содействия нормальным трудовым отношениям и создания эффективно действующих систем инспекции труда;',
 'Г-н\xa0Алкалай (Б

In [23]:

raw_datasets["train"][:14]["dest_text"]

['《宣言》可以提供必要的框架，用于召集地方、国家和区域三级的土著人民组织，集体取得人权成果。',
 '在这方面，我们只需提到前南斯拉夫问题国际刑事法庭、卢旺达问题国际刑事法庭、国际海洋法法庭和国际刑事法院。',
 '主席（以俄语发言）：有15票赞成。',
 '这就是为什么我们必须迅速拿起榔头和钉子，改造马蹄型会议桌。 二十一世纪不需要马蹄型会议桌，而是需要圆形桌，可以多放几把椅子。',
 '此后，应每隔五年进行核查和核证直至入计期结束。',
 '31所高等教育院校设有俄语和俄罗斯文学培训课程。',
 '- 使劳动法和机构富有成效，包括有关承认雇佣关系、促进良好的产业关系以及建立有效的劳动监察制度；和',
 '阿尔卡拉伊先生（波斯尼亚和黑塞哥维那）（以英语发言）：今天，我非常荣幸能与诸位一起在此开会，我要借此机会由衷地感谢联合国大会主席及菲律宾和巴基斯坦两国政府召开这次会议，讨论这一重要议题。',
 'WFP还参加UNSCN关于HIV/AIDS、家庭粮食安全、学校保健与营养、紧急情况中的营养和微量营养素等问题工作组的工作。',
 '15. 确认秘书长的斡旋在非洲起着重要作用，并鼓励秘书长继续尽可能经常运用调解手段来帮助和平解决冲突，并在这方面酌情与非洲联盟和其他次区域组织进行密切协作；',
 '2000年5月31日伊拉克代表给秘书长的信（S/2000/528）。',
 '全球环境基金理事会于2003年11月在华盛顿举行了会议，在会上请环境基金的首席执行官向理事会提交一份建议草案，以供审查和发表评论，提交的时间应足够提前，以便能够把理事会的意见反映在定于2005年提交第七届缔约国会议的谅解备忘录草稿之中。',
 '回历1424年3月27日-29日(2003年5月28日-30日)于伊朗伊斯兰共和国德黑兰举行的伊斯兰外交部长第三十届会议(团结与尊严会议)，',
 '四、信息和宣传']

In [24]:
raw_datasets["train"][:14]["dest_lang"]

['zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh']

We have prepared the dataset to contain Russian source texts and Chinese target texts.

Since we have already selected the specific language pair (ru-zh), we don't need to perform additional filtering by language.

Now we load the pre-trained tokenizer for the NLLB model and apply it to the Russian-Chinese pair:

In [25]:
max_tok_length = 128

from transformers import AutoTokenizer

checkpoint = "facebook/nllb-200-distilled-600M"
# from flores200_codes import flores_codes
src_code = "rus_Cyrl"
tgt_code = "zho_Hans"
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint, 
    padding=True, 
    pad_to_multiple_of=8, 
    src_lang=src_code, 
    tgt_lang=tgt_code, 
    truncation=True, 
    max_length=max_tok_length,
    )

We can apply the tokenizer function to any dataset taking advantage that Hugging Face Datasets are [Apache Arrow](https://arrow.apache.org) files stored on the disk, so you only keep the samples you ask for loaded in memory.

To keep the data as a dataset, we will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This also allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the training needs of the model that is to be used:

In [26]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["source_text"], 
        text_target = sample["dest_text"],
        )
    return model_inputs


The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*. We can check what the preprocess_function is doing with a small sample

In [27]:
sample = raw_datasets["train"].select(range(2))
model_input = preprocess_function({
    "source_text": list(sample["source_text"]),
    "dest_text": list(sample["dest_text"]),
})
print(model_input)

{'input_ids': [[256147, 1643, 6130, 67230, 23064, 722, 213, 108213, 567, 104246, 55184, 222031, 16632, 203222, 166760, 6618, 255, 16881, 114768, 64892, 567, 507, 6335, 5824, 23739, 16632, 28531, 537, 6409, 89293, 110096, 248079, 213, 112997, 13072, 71105, 885, 131045, 177648, 2171, 35137, 4900, 147601, 30746, 191, 82357, 2060, 180968, 507, 6335, 5824, 213, 11038, 6335, 5824, 23739, 16632, 248075, 2], [256147, 7628, 722, 110096, 60014, 885, 191, 73849, 134758, 172241, 55191, 11344, 13946, 4091, 14587, 366, 3245, 248128, 66499, 19174, 343, 10221, 239405, 248079, 172241, 55191, 11344, 84690, 19168, 11344, 13946, 4091, 14587, 366, 13988, 530, 489, 248079, 172241, 55191, 11344, 13946, 4091, 14587, 366, 75414, 241149, 200068, 248079, 172241, 55191, 11344, 84690, 19168, 11344, 36750, 248075, 2]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [

In [28]:
for sample in model_input['input_ids']:
    print(tokenizer.convert_ids_to_tokens(sample))

['rus_Cyrl', '▁У', 'про', 'щения', '▁тек', 'ста', '▁и', '▁подготов', 'ки', '▁пря', 'мых', '▁перево', 'дов', '▁Дек', 'лара', 'ции', '▁на', '▁разли', 'чные', '▁язы', 'ки', '▁ко', 'рен', 'ных', '▁наро', 'дов', '▁будет', '▁я', 'вно', '▁недоста', 'точно', ',', '▁и', '▁потребу', 'ется', '▁приня', 'ть', '▁другие', '▁меры', '▁для', '▁созда', 'ния', '▁потенци', 'ала', '▁в', '▁рамках', '▁об', 'щин', '▁ко', 'рен', 'ных', '▁и', '▁неко', 'рен', 'ных', '▁наро', 'дов', '.', '</s>']
['rus_Cyrl', '▁До', 'ста', 'точно', '▁назва', 'ть', '▁в', '▁этой', '▁связи', '▁Между', 'народ', 'ный', '▁три', 'бу', 'нал', '▁по', '▁бы', 'в', 'шей', '▁Ю', 'го', 'сла', 'вии', ',', '▁Между', 'народ', 'ный', '▁уго', 'лов', 'ный', '▁три', 'бу', 'нал', '▁по', '▁Ру', 'ан', 'де', ',', '▁Между', 'народ', 'ный', '▁три', 'бу', 'нал', '▁по', '▁мор', 'скому', '▁праву', ',', '▁Между', 'народ', 'ный', '▁уго', 'лов', 'ный', '▁суд', '.', '</s>']


We can recover the source text by applying [batch_decode](https://huggingface.co/docs/transformers/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode) of the tokenizer 

In [29]:
tokenizer.batch_decode(model_input['input_ids'])

['rus_Cyrl Упрощения текста и подготовки прямых переводов Декларации на различные языки коренных народов будет явно недостаточно, и потребуется принять другие меры для создания потенциала в рамках общин коренных и некоренных народов.</s>',
 'rus_Cyrl Достаточно назвать в этой связи Международный трибунал по бывшей Югославии, Международный уголовный трибунал по Руанде, Международный трибунал по морскому праву, Международный уголовный суд.</s>']

Now, we can apply the preprocess_function to the raw datasets (training, validation and test):

In [30]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

We are going to filter the tokenized datasets by maximum number of tokens in source and target language:

In [31]:
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) <= max_tok_length and len(x["labels"]) <= max_tok_length , desc=f"Discarding source and target sentences with more than {max_tok_length} tokens")

Discarding source and target sentences with more than 128 tokens:   0%|          | 0/10000 [00:00<?, ? example…

Discarding source and target sentences with more than 128 tokens:   0%|          | 0/1000 [00:00<?, ? examples…

Discarding source and target sentences with more than 128 tokens:   0%|          | 0/1000 [00:00<?, ? examples…

We can take a quick look at the length histogram in the source language:

In [32]:
dic = {}
for sample in tokenized_datasets['train']:
    sample_length = len(sample['input_ids'])
    if sample_length not in dic:
        dic[sample_length] = 1
    else:
        dic[sample_length] += 1 

for i in range(1,max_tok_length+1):
    if i in dic:
        print(f"{i:>2} {dic[i]:>3}")

 3  23
 4  65
 5 180
 6 142
 7 151
 8 125
 9 160
10 120
11 132
12 138
13 160
14 169
15 121
16 122
17 146
18 130
19 108
20 120
21 161
22 180
23 126
24 140
25 141
26 141
27 141
28 161
29 166
30 166
31 155
32 150
33 142
34 158
35 161
36 153
37 153
38 149
39 133
40 137
41 148
42 133
43 137
44 128
45 117
46 129
47 138
48 125
49 127
50 106
51 122
52  98
53 114
54 106
55  97
56 116
57  99
58 112
59  96
60 100
61 101
62  80
63  73
64  93
65  61
66  85
67  75
68  67
69  50
70  79
71  65
72  64
73  50
74  73
75  54
76  45
77  35
78  51
79  44
80  44
81  33
82  46
83  34
84  28
85  38
86  33
87  25
88  28
89  35
90  27
91  33
92  24
93  25
94  26
95  19
96  23
97  22
98  13
99  20
100  20
101  20
102   8
103  14
104  17
105  10
106  14
107  11
108  12
109   5
110  12
111   7
112  13
113   6
114  13
115   5
116   6
117  13
118   5
119   5
120   6
121   8
122   7
123   7
124   6
125   8
126   6
127   6
128   7


Checking a sample after filtering by maximum number of tokens:

In [33]:
for sample in tokenized_datasets['train'].select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[256147, 1643, 6130, 67230, 23064, 722, 213, 108213, 567, 104246, 55184, 222031, 16632, 203222, 166760, 6618, 255, 16881, 114768, 64892, 567, 507, 6335, 5824, 23739, 16632, 28531, 537, 6409, 89293, 110096, 248079, 213, 112997, 13072, 71105, 885, 131045, 177648, 2171, 35137, 4900, 147601, 30746, 191, 82357, 2060, 180968, 507, 6335, 5824, 213, 11038, 6335, 5824, 23739, 16632, 248075, 2]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[256200, 248059, 3, 45461, 3, 10906, 10757, 5528, 248506, 198410, 248079, 93281, 252162, 250227, 28767, 253131, 13168, 249249, 45023, 250447, 251916, 248506, 250646, 250621, 53280, 28973, 248079, 250227, 249601, 73484, 59192, 67456, 253935, 2]
[256147, 7628, 722, 110096, 60014, 885, 191, 73849, 134758, 172241, 55191, 11344, 13946, 4091, 14587, 366, 3245, 248128, 66499, 19174, 343, 10221, 239405, 248079, 172241, 55191, 11344, 84690

bitsandbytes is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>


In [34]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [35]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    quantization_config=quantization_config
    )


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

## Evaluation

The last thing to define for our Seq2SeqTrainer is how to compute the metrics to evaluate the predictions of our model with respect to references. To this purpose, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu). You can see a simple example of usage below:

:

In [36]:
from evaluate import load

metric = load("sacrebleu")

We need to define a function compute_metrics to compute BLEU scores at each epoch. The example below performs a basic post-processing to decode the predictions into texts:

In [37]:
import numpy as np
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Convert to lists if coming from a datasets.Column
    if not isinstance(labels, list):
        labels = list(labels)
        
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace negative ids in the labels as we can't decode them.
    labels = [
        [tokenizer.pad_token_id if j < 0 else j for j in label]
        for label in labels
    ]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

## Inference

At inference time, it is recommended to use the [generate function](https://huggingface.co/docs/transformers/main_classes/text_generation). This method takes care of encoding the input and auto-regressively generates the decoder output. Check out [this blog post](https://huggingface.co/blog/how-to-generate) to know all the details about generating text with Transformers.
There’s also [this blog post](https://huggingface.co/blog/encoder-decoder#encoder-decoder) which explains how generation works in general in encoder-decoder models.

Let us first load the default inference parameters of NLLB.

In [38]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
)

print(generation_config)

GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "eos_token_id": 2,
  "max_length": 200,
  "pad_token_id": 1
}



We prepare the test set in batches to be translated:

In [39]:
test_batch_size = 32
batch_tokenized_test = tokenized_datasets['test'].batch(test_batch_size)

Batching examples:   0%|          | 0/983 [00:00<?, ? examples/s]

Processing in batches to add padding and convert to tensors, then perform inference with num_beams = 1 and do_sample = False, that is, greedy search.

In [40]:
number_of_batches = len(batch_tokenized_test["source_text"])
output_sequences = []
for i in range(number_of_batches):
    inputs = tokenizer(
        batch_tokenized_test["source_text"][i], 
        max_length=max_tok_length, 
        truncation=True, 
        return_tensors="pt", 
        padding=True,
        )
    with torch.no_grad():    
        output_batch = model.generate(
            generation_config=generation_config, 
            input_ids=inputs["input_ids"].cuda(), 
            attention_mask=inputs["attention_mask"].cuda(), 
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_code), 
            max_length = max_tok_length, 
            num_beams=1, 
            do_sample=False,
            )
    output_sequences.extend(output_batch.cpu())

In [ ]:
# Calcular BLEU
bleu_result = compute_metrics((output_sequences, tokenized_datasets["test"]["labels"]))

# Preparar datos para COMET
decoded_preds = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(tokenized_datasets["test"]["labels"], skip_special_tokens=True)
source_texts = tokenized_datasets["test"]["source_text"]

# COMET requiere: source, hypothesis (predictions), reference
comet_input = []
for src_text, pred_text, ref_text in zip(source_texts, decoded_preds, decoded_labels):
    comet_input.append({
        "src": src_text,
        "mt": pred_text.strip(),
        "ref": ref_text.strip()
    })

# Calcular COMET
comet_result = comet_model.predict(comet_input, batch_size=8, gpus=1)

print(f'BLEU score: {bleu_result["bleu"]:.2f}')
print(f'COMET score: {comet_result["system_score"]:.4f}')

BLEU score: 12.3266


In [ ]:
from comet import download_model, load_from_checkpoint

# Cargar COMET (modelo recomendado: Unbabel/wmt22-comet-da)
comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_model_path)